# Notebook 01A: Set Up the Team SageMaker MLflow App

**Module:** ITI113 Machine Learning and Operations  
**Focus Area:** C, MLOps and Deployment (Experiment Tracking Setup)  

---

## What this notebook does

1. Checks whether the installed AWS SDK supports SageMaker MLflow App APIs.
2. Sets the course/team/student/project configuration.
3. Creates or reuses a SageMaker MLflow App.
4. Creates the MLflow App with these tags when it does not already exist: Course, Semester, TeamId, StudentId, ProjectName, CreatedByNotebook.
5. If the App already exists, checks its tags and tries to add or update the required tags.
6. Prints the MLflow App ARN, which becomes the MLflow tracking URI.
7. Creates a presigned MLflow UI URL.
8. Runs a small MLflow test run using the configured team and student metadata.

Expected result:

- MLflow tracking URI: arn:aws:sagemaker:ap-southeast-1:<account-id>:mlflow-app/...
- Experiment: ITI113/<team-id>/Experiment1
- Run name: <team-id>_<student-id>_mlflow_app_test


## 1. Install or update required packages

The sagemaker-mlflow plugin lets the standard MLflow client authenticate to SageMaker MLflow using AWS IAM SigV4, so no separate MLflow credentials are needed.

In [1]:
%%capture
%pip install -U boto3 botocore mlflow sagemaker-mlflow
print("Packages installed")


## 2. Configuration

All downstream names are derived from TEAM_ID, so only the team, student and project values need changing. For team03 the execution role resolves to arn:aws:iam::<account-id>:role/SageMakerExecutionRole-ITI113-Team03, which must be able to write to the S3 artifact store.

In [2]:
import os
os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"

import warnings
warnings.filterwarnings("ignore")

import boto3
from botocore.exceptions import ClientError
import time
import json
from datetime import datetime
from pathlib import Path

REGION = "ap-southeast-1"
COURSE = "ITI113"
SEMESTER = "26S1"

# Change these for each team/student.
TEAM_ID = "team03"
STUDENT_ID = "s301"

# Project name used in tags and artifact organisation.
PROJECT_NAME = "crypto-scam-detector"

# Existing course bucket from the SageMaker Pipeline lab.
CLASS_BUCKET = "nyp-26s1-iti113"

# One MLflow App per team is usually enough.
MLFLOW_APP_NAME = f"iti113-26s1-{TEAM_ID}-mlflow-app"

# Where MLflow run artifacts will be stored.
ARTIFACT_STORE_URI = f"s3://{CLASS_BUCKET}/iti113/{TEAM_ID}/mlflow-app-artifacts/"

# Experiment name inside MLflow App. Use a normal MLflow experiment name, not a Databricks /Workspace path.
EXPERIMENT_NAME = f"{COURSE}/{TEAM_ID}/Experiment1"

# Optional: make this MLflow App the account/domain default. Keep False for classroom safety.
SET_AS_ACCOUNT_DEFAULT = False
SET_AS_DEFAULT_FOR_EXISTING_DOMAINS = False

session = boto3.Session(region_name=REGION)
sts = session.client("sts")
sm = session.client("sagemaker")
s3 = session.client("s3")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
CALLER_ARN = sts.get_caller_identity()["Arn"]

TEAM_ROLE_SUFFIX = TEAM_ID.lower().replace("team", "Team")
ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/SageMakerExecutionRole-ITI113-{TEAM_ROLE_SUFFIX}"

# Required tags for team-level MLflow IAM restriction.
MLFLOW_APP_TAGS = [
    {"Key": "Course", "Value": COURSE},
    {"Key": "Semester", "Value": SEMESTER},
    {"Key": "TeamId", "Value": TEAM_ID},
    {"Key": "StudentId", "Value": STUDENT_ID},
    {"Key": "ProjectName", "Value": PROJECT_NAME},
    {"Key": "CreatedByNotebook", "Value": "01A_setup_sagemaker_mlflow_app"},
]

print("Account:", ACCOUNT_ID)
print("Caller ARN:", CALLER_ARN)
print("Region:", REGION)
print("Team ID:", TEAM_ID)
print("Student ID:", STUDENT_ID)
print("Project Name:", PROJECT_NAME)
print("MLflow App Name:", MLFLOW_APP_NAME)
print("Artifact Store:", ARTIFACT_STORE_URI)
print("Role ARN:", ROLE_ARN)
print("Experiment:", EXPERIMENT_NAME)
print("\nMLflow App tags to apply:")
for tag in MLFLOW_APP_TAGS:
    print(f"  {tag['Key']} = {tag['Value']}")


Account: 044528205969
Caller ARN: arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team03/SageMaker
Region: ap-southeast-1
Team ID: team03
Student ID: s301
Project Name: crypto-scam-detector
MLflow App Name: iti113-26s1-team03-mlflow-app
Artifact Store: s3://nyp-26s1-iti113/iti113/team03/mlflow-app-artifacts/
Role ARN: arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team03
Experiment: ITI113/team03/Experiment1

MLflow App tags to apply:
  Course = ITI113
  Semester = 26S1
  TeamId = team03
  StudentId = s301
  ProjectName = crypto-scam-detector
  CreatedByNotebook = 01A_setup_sagemaker_mlflow_app


## 3. Check SDK/API support

Fails fast if the installed boto3 or botocore version does not support the SageMaker MLflow App APIs, rather than failing part-way through creating resources.

In [3]:
required_methods = [
    "create_mlflow_app",
    "list_mlflow_apps",
    "describe_mlflow_app",
    "create_presigned_mlflow_app_url",
    "add_tags",
    "list_tags",
]

missing = [method for method in required_methods if not hasattr(sm, method)]

print("SageMaker client supports:")
for method in required_methods:
    print(f"  {method}: {hasattr(sm, method)}")

if missing:
    raise RuntimeError(
        "Your boto3/botocore version does not support these SageMaker MLflow App/tag APIs: "
        + ", ".join(missing)
        + "\nRun the package update cell, restart the kernel, and rerun."
    )


SageMaker client supports:
  create_mlflow_app: True
  list_mlflow_apps: True
  describe_mlflow_app: True
  create_presigned_mlflow_app_url: True
  add_tags: True
  list_tags: True


## 4. Verify the S3 artifact store prefix is writable

This checks that the current role can write a small test file to the MLflow artifact prefix.


In [4]:
from urllib.parse import urlparse

def parse_s3_uri(uri: str):
    parsed = urlparse(uri)
    if parsed.scheme != "s3":
        raise ValueError(f"Not an S3 URI: {uri}")
    return parsed.netloc, parsed.path.lstrip("/")

artifact_bucket, artifact_prefix = parse_s3_uri(ARTIFACT_STORE_URI)
test_key = f"{artifact_prefix.rstrip('/')}/_setup_test/{TEAM_ID}_{STUDENT_ID}_write_test.txt"

try:
    s3.put_object(
        Bucket=artifact_bucket,
        Key=test_key,
        Body=(
            f"MLflow App setup write test at {datetime.utcnow().isoformat()}Z\n"
        ).encode("utf-8"),
    )
    print("S3 write test succeeded:")
    print(f"s3://{artifact_bucket}/{test_key}")
except ClientError:
    print("S3 write test failed. Check bucket/prefix permissions.")
    raise


S3 write test succeeded:
s3://nyp-26s1-iti113/iti113/team03/mlflow-app-artifacts/_setup_test/team03_s301_write_test.txt


## 5. Optionally discover SageMaker Studio domain IDs

This is only needed if you want the MLflow App to be configured as the default for one or more Studio domains. For the classroom, it is safer to leave SET_AS_DEFAULT_FOR_EXISTING_DOMAINS set to False.


In [5]:
domain_ids = []

try:
    domains_response = sm.list_domains()
    domain_ids = [domain["DomainId"] for domain in domains_response.get("Domains", [])]
    print("Studio domains found:", domain_ids)
except ClientError as e:
    print("Could not list Studio domains. This is okay if you are not setting defaults.")
    print(e)


Studio domains found: ['d-popmr5pqbh1n', 'd-wcuptup1pj6w', 'd-gpdrdk2w4dgw', 'd-5q0cdlsfisve', 'd-lsrccj8tewjf', 'd-oijl6tgx8os2', 'd-cydcdxkc4yot', 'd-tjsofl2bch3a', 'd-8nb3rzhhmygx', 'd-jgu4uwvdu9ir']


## 6. Create or reuse the SageMaker MLflow App

Idempotent by design: the cell looks for an App with the configured name first, so re-running this notebook reuses the existing App rather than creating duplicates on the shared class account.

- If the App does not exist, it is created with the required tags.
- If it already exists, the tags are checked and any missing or incorrect ones are re-applied.

The tags drive team-level IAM restriction, since team03's role is permitted only where the ResourceTag TeamId equals team03.

Creation is asynchronous, so the next cell polls until the status reaches Created or Updated.

In [6]:
def find_mlflow_app_by_name(name: str):
    paginator = sm.get_paginator("list_mlflow_apps")
    for page in paginator.paginate():
        for summary in page.get("Summaries", []):
            if summary.get("Name") == name:
                return summary
    return None


def ensure_mlflow_app_tags(resource_arn: str, required_tags: list):
    """Check and apply required tags to an existing MLflow App.

    If the current role does not have sagemaker:AddTags/ListTags permission,
    this function will print a warning and continue. The admin can tag the App later.
    """
    required = {tag["Key"]: tag["Value"] for tag in required_tags}

    try:
        existing_tags_response = sm.list_tags(ResourceArn=resource_arn)
        existing = {
            tag["Key"]: tag["Value"]
            for tag in existing_tags_response.get("Tags", [])
        }

        missing_or_different = [
            {"Key": key, "Value": value}
            for key, value in required.items()
            if existing.get(key) != value
        ]

        if missing_or_different:
            print("\nAdding/updating required MLflow App tags:")
            for tag in missing_or_different:
                print(f"  {tag['Key']} = {tag['Value']}")

            sm.add_tags(
                ResourceArn=resource_arn,
                Tags=missing_or_different,
            )
        else:
            print("\nExisting MLflow App already has the required tags.")

        final_tags = sm.list_tags(ResourceArn=resource_arn).get("Tags", [])
        print("\nCurrent MLflow App tags:")
        for tag in final_tags:
            print(f"  {tag['Key']} = {tag['Value']}")

    except ClientError as e:
        print("\n[WARNING] Could not verify or update MLflow App tags.")
        print("This may happen if the current role does not have sagemaker:ListTags/AddTags.")
        print("Ask the admin to ensure these tags exist on the MLflow App:")
        for tag in required_tags:
            print(f"  {tag['Key']} = {tag['Value']}")
        print("\nOriginal error:")
        print(e)


existing = find_mlflow_app_by_name(MLFLOW_APP_NAME)

if existing:
    mlflow_app_arn = existing["Arn"]
    print("Reusing existing MLflow App:")
    print(json.dumps(existing, indent=2, default=str))

    # Important for team-level MLflow IAM restriction.
    ensure_mlflow_app_tags(mlflow_app_arn, MLFLOW_APP_TAGS)

else:
    create_args = {
        "Name": MLFLOW_APP_NAME,
        "ArtifactStoreUri": ARTIFACT_STORE_URI,
        "RoleArn": ROLE_ARN,
        "ModelRegistrationMode": "AutoModelRegistrationDisabled",
        "Tags": MLFLOW_APP_TAGS,
    }

    if SET_AS_ACCOUNT_DEFAULT:
        create_args["AccountDefaultStatus"] = "ENABLED"

    if SET_AS_DEFAULT_FOR_EXISTING_DOMAINS and domain_ids:
        create_args["DefaultDomainIdList"] = domain_ids

    print("Creating MLflow App with args:")
    print(json.dumps(create_args, indent=2, default=str))

    response = sm.create_mlflow_app(**create_args)
    mlflow_app_arn = response["Arn"]
    print("Create response:", response)

print("\nMLflow App ARN:")
print(mlflow_app_arn)


Reusing existing MLflow App:
{
  "Arn": "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-J5AYUG4AJHVW",
  "Name": "iti113-26s1-team03-mlflow-app",
  "Status": "Created",
  "CreationTime": "2026-07-26 05:40:04+00:00",
  "LastModifiedTime": "2026-08-04 18:04:19.341000+00:00",
  "MlflowVersion": "3.10.1"
}

Existing MLflow App already has the required tags.



Current MLflow App tags:
  Semester = 26S1
  sagemaker:domain-arn = arn:aws:sagemaker:ap-southeast-1:044528205969:domain/d-gpdrdk2w4dgw
  ProjectName = crypto-scam-detector
  sagemaker:space-arn = arn:aws:sagemaker:ap-southeast-1:044528205969:space/d-gpdrdk2w4dgw/team03-shared
  Course = ITI113
  TeamId = team03
  CreatedByNotebook = 01A_setup_sagemaker_mlflow_app
  StudentId = s301

MLflow App ARN:
arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-J5AYUG4AJHVW


In [7]:
def wait_for_mlflow_app(arn: str, timeout_seconds: int = 600, poll_seconds: int = 20):
    start = time.time()
    last_status = None

    while True:
        desc = sm.describe_mlflow_app(Arn=arn)
        status = desc.get("Status")

        if status != last_status:
            print(f"Status: {status}")
            last_status = status

        if status in ["Created", "Updated"]:
            return desc

        if status in ["CreateFailed", "UpdateFailed", "DeleteFailed", "Deleted"]:
            raise RuntimeError(
                f"MLflow App entered failure status: {status}\n"
                + json.dumps(desc, indent=2, default=str)
            )

        if time.time() - start > timeout_seconds:
            raise TimeoutError(f"Timed out waiting for MLflow App. Last status: {status}")

        time.sleep(poll_seconds)

mlflow_app_desc = wait_for_mlflow_app(mlflow_app_arn)

print("\nFinal MLflow App description:")
print(json.dumps(mlflow_app_desc, indent=2, default=str))


Status: Created

Final MLflow App description:
{
  "Arn": "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-J5AYUG4AJHVW",
  "Name": "iti113-26s1-team03-mlflow-app",
  "ArtifactStoreUri": "s3://nyp-26s1-iti113/iti113/team03/mlflow-app-artifacts/",
  "MlflowVersion": "3.10.1",
  "RoleArn": "arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team03",
  "Status": "Created",
  "ModelRegistrationMode": "AutoModelRegistrationDisabled",
  "CreationTime": "2026-07-26 05:40:04+00:00",
  "CreatedBy": {
    "IamIdentity": {
      "Arn": "arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team03/SageMaker"
    }
  },
  "LastModifiedTime": "2026-08-04 18:04:19.341000+00:00",
  "LastModifiedBy": {
    "IamIdentity": {
      "Arn": "arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team03/SageMaker"
    }
  },
  "WeeklyMaintenanceWindowStart": "Tue:18:33",
  "MaintenanceStatus": "MAINTENANCE_COMPLETE",
  "ResponseMetadata": {
    "RequestId":

## 7. Get a presigned MLflow UI URL

Access to the MLflow UI is granted through a short-lived presigned URL (300-second expiry) rather than standing credentials.

**Note:** this cell's output is intentionally cleared before committing. The presigned URL embeds a temporary authentication token, which should not be stored in version control.

In [8]:
url_response = sm.create_presigned_mlflow_app_url(
    Arn=mlflow_app_arn,
    ExpiresInSeconds=300,
    SessionExpirationDurationInSeconds=3600,
)

mlflow_ui_url = url_response["AuthorizedUrl"]

print("Open this MLflow UI URL in a browser tab:")
print(mlflow_ui_url)


Open this MLflow UI URL in a browser tab:
https://app-J5AYUG4AJHVW.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkRSVFNCTSIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNFcvWDFHL3Iza2w4QnRzbHMrYnVRZiswRjZMVjB1bzBOaHJyWEtaaWprWDhBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGd2QwOXllRWR3VXpORGRUWm9SMFoxVldsb1VXNUtlbTVuTm10bGJXbzJMM0o1Tm5kT1kxSkRNRkIzTVV0clJHTmpaRkZHWTBGMU9ITTRNMUZrTlRKVVVUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFiVHJFZnIzaGF5OUlCeFl5NnBrWFhZQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF3elppaGNScjRaUHZ0RVVUTUNBUkNBTzJ4U2J2eWJ3ZEZMMEdpTkpQME80Z25aZlVEYkVDT1V0US81VmdMLzhMc2lrb1p6cTNrR0hIZ3NRa1NnQk0rSzg4QklKaDZUbE1sOTc3TC9BZ0FBRUFCSC9jWVdBSVI1aEd0N24vVmxIVFB0Y0VXSU1UVGJLRkZmRk1Pcmthak1RVHVNT

## 8. Test MLflow logging against the SageMaker MLflow App

This uses the SageMaker MLflow App ARN as the MLflow tracking URI. No Databricks host or token is required.


In [9]:
import mlflow
import tempfile
from pathlib import Path

print("MLflow version:", mlflow.__version__)

mlflow.set_tracking_uri(mlflow_app_arn)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f"{TEAM_ID}_{STUDENT_ID}_mlflow_app_test"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.set_tags({
        "course": COURSE,
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "tracking_backend": "sagemaker_mlflow_app",
        "purpose": "setup_validation",
        "project_name": PROJECT_NAME,
    })

    mlflow.log_params({
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "region": REGION,
        "artifact_store_uri": ARTIFACT_STORE_URI,
        "project_name": PROJECT_NAME,
    })

    mlflow.log_metrics({
        "setup_smoke_test": 1.0,
    })

    summary = {
        "message": "SageMaker MLflow App logging test succeeded.",
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "project_name": PROJECT_NAME,
        "mlflow_app_arn": mlflow_app_arn,
        "experiment_name": EXPERIMENT_NAME,
        "run_id": run.info.run_id,
    }

    with tempfile.TemporaryDirectory() as tmpdir:
        artifact_path = Path(tmpdir) / "mlflow_app_setup_summary.json"
        artifact_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
        mlflow.log_artifact(str(artifact_path), artifact_path="setup_test")

    run_id = run.info.run_id

print("Logged test run successfully.")
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", run_name)
print("Run ID:", run_id)


MLflow version: 3.15.1


🏃 View run team03_s301_mlflow_app_test at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/d6da354281bf4398b833713246866871
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
Logged test run successfully.
Experiment: ITI113/team03/Experiment1
Run name: team03_s301_mlflow_app_test
Run ID: d6da354281bf4398b833713246866871


## 9. Search the test run

This confirms that the run is visible through the MLflow API.


In [10]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    raise RuntimeError(f"Experiment not found: {EXPERIMENT_NAME}")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.student_id = '{STUDENT_ID}' and tags.team_id = '{TEAM_ID}'",
    order_by=["attributes.start_time DESC"],
    max_results=10,
)

cols = ["run_id", "tags.mlflow.runName", "metrics.setup_smoke_test",
        "metrics.test_auc_roc", "metrics.test_f1", "metrics.test_accuracy"]
runs[[c for c in cols if c in runs.columns]]


,run_id,tags.mlflow.runName,metrics.setup_smoke_test,metrics.test_auc_roc,metrics.test_f1,metrics.test_accuracy
0,d6da354281bf4398b833713246866871,team03_s301_mlflow_app_test,1.0,NaN,NaN,NaN
1,c3b62f67fed348c8b1224098aa445ede,team03_s301_mlflow_app_test,1.0,NaN,NaN,NaN
2,ec49e9ce80304a499538856d1a1f39b3,team03_s301_mlflow_app_test,1.0,NaN,NaN,NaN
3,c52addc62ba54379bafd565608b15187,team03_s301_mlflow_app_test,1.0,NaN,NaN,NaN
4,1e3189fccb344a1fbea26b406554f652,team03_s301_mlflow_app_test,1.0,NaN,NaN,NaN
5,f8b464cb5eee4bf586fb7bb8df871f14,team03_s301_mlflow_app_test,1.0,NaN,NaN,NaN
6,8e66cda08af54619a42167cb20c54eee,team03_s301_mlflow_app_test,1.0,NaN,NaN,NaN
7,0e3c0458a9154927977b9f2c8f209c46,team03_s301_mlflow_app_test,1.0,NaN,NaN,NaN
8,12306d4558044a4e91f8541a22e5779b,rf_candidate_05,NaN,0.8600,0.6718,0.7373
9,2cdbd0fa7f8844c5814fe6048f2baae8,rf_candidate_04,NaN,0.8385,0.6486,0.7206


## 10. Output values to copy into future notebooks


In [11]:
print("# Copy these into future notebooks")
print(f'REGION = "{REGION}"')
print(f'MLFLOW_APP_ARN = "{mlflow_app_arn}"')
print(f'EXPERIMENT_NAME = "{EXPERIMENT_NAME}"')
print(f'TEAM_ID = "{TEAM_ID}"')
print(f'STUDENT_ID = "{STUDENT_ID}"')
print(f'PROJECT_NAME = "{PROJECT_NAME}"')

# Save locally for convenience.
config = {
    "REGION": REGION,
    "MLFLOW_APP_ARN": mlflow_app_arn,
    "EXPERIMENT_NAME": EXPERIMENT_NAME,
    "TEAM_ID": TEAM_ID,
    "STUDENT_ID": STUDENT_ID,
    "PROJECT_NAME": PROJECT_NAME,
    "ARTIFACT_STORE_URI": ARTIFACT_STORE_URI,
    "MLFLOW_APP_NAME": MLFLOW_APP_NAME,
    "MLFLOW_APP_TAGS": MLFLOW_APP_TAGS,
}

savepath = f"mlflow_app_config_{TEAM_ID.lower()}_{STUDENT_ID.lower()}.json"

Path(savepath).write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)
print(f"\nSaved local config: {savepath}")


# Copy these into future notebooks
REGION = "ap-southeast-1"
MLFLOW_APP_ARN = "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-J5AYUG4AJHVW"
EXPERIMENT_NAME = "ITI113/team03/Experiment1"
TEAM_ID = "team03"
STUDENT_ID = "s301"
PROJECT_NAME = "crypto-scam-detector"

Saved local config: mlflow_app_config_team03_s301.json
